# 🧩 Documentos para RAG

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 3 — 35 minutos**

---

## Objetivo

Entender el primer paso de RAG: cargar documentos de distintas fuentes y dividirlos en fragmentos coherentes (chunking).

## Al terminar este bloque vas a poder:

1. Cargar documentos PDF y páginas web usando LangChain.
2. Inspeccionar la estructura de un Document (page_content + metadata).
3. Explicar por qué el chunking es necesario antes de vectorizar.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **RAG** | Retrieval-Augmented Generation: buscar contexto relevante antes de generar una respuesta. |
| **Document Loader** | Componente que abre un archivo (PDF, web, etc.) y extrae su texto. |
| **Chunking** | Dividir un documento largo en fragmentos pequeños para poder vectorizarlos. |
| **Overlap** | Repetir algunos tokens del final de un chunk al inicio del siguiente para no perder contexto. |
| **page_content / metadata** | Las dos partes de todo Document en LangChain: el texto y la información contextual. |

In [ ]:
# Instalación de librerías necesarias
# LangChain es la librería principal para construir aplicaciones RAG
!pip install langchain -q

print("Instalación completada")

## ¿Qué es RAG y por qué empieza por los documentos?

### Analogía

Imaginá que tenés que responder un examen con libro abierto. Antes de escribir cualquier cosa, buscás en el índice, encontrás las páginas relevantes y las marcás. Eso es RAG: primero *recuperás* el contexto específico, después *generás* la respuesta fundamentada en él.

### Dónde vive esto en el mundo real

Los sistemas de atención al cliente de bancos, aseguradoras y empresas de software usan RAG para responder con información de sus manuales internos. El modelo nunca memorizó esos documentos — los busca en tiempo real. Eso significa que podés actualizar la documentación sin reentrenar nada.

### El pipeline RAG completo (tres bloques de hoy)

| Paso | Qué hace | Dónde lo vemos |
|---|---|---|
| **1. Cargar** | Extraer texto de PDFs, webs, etc. | Este notebook |
| **2. Vectorizar** | Convertir chunks en embeddings y guardarlos en ChromaDB | Siguiente |
| **3. Generar** | Recuperar contexto relevante + responder con Gemini | Último |


### ✎ Para pensar

- ¿Por qué no podemos simplemente pegarle todo el documento al LLM en lugar de fragmentarlo?
- Si un documento tiene 500 páginas, ¿cuántos chunks aproximados esperarías con chunk_size=500 caracteres?

## Carga de documentos PDF

PyPDFLoader convierte cada página del PDF en un objeto `Document` con dos campos:

- `page_content`: el texto de la página
- `metadata`: diccionario con fuente, número de página, etc.

Esta estructura uniforme es la que ChromaDB va a recibir en el siguiente notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado correctamente en /content/drive")

In [ ]:
# Instalamos la librería para trabajar con PDFs
# PyPDF2 es la librería más popular para extraer texto de archivos PDF
!pip install pypdf -q

print("PyPDF instalado correctamente")

In [ ]:
!pip install -U langchain-community -q
print("langchain-community instalado correctamente")

In [ ]:
# Cargamos un PDF usando PyPDFLoader de LangChain
from langchain.document_loaders import PyPDFLoader

# Especificamos la ruta del archivo PDF
# En un caso real, aquí pondrías la ruta a tu propio documento
loader = PyPDFLoader("/content/drive/MyDrive/Clases/000 - 2DO Cuatrimestre/HABLA/011 - LLMs APIS RAG/generacion_de_imagenes.pdf")

# Cargamos todas las páginas del PDF
pages = loader.load()

print(f"PDF cargado exitosamente: {len(pages)} páginas")
print("Cada página se convierte en un objeto Document independiente")

In [ ]:
# Verificamos cuántas páginas se cargaron
len(pages)

In [ ]:
# Examinamos la primera página como ejemplo
page = pages[0]
print("Primera página seleccionada para análisis")

In [ ]:
# Mostramos los primeros 500 caracteres del contenido de la página
print("CONTENIDO DE LA PRIMERA PÁGINA (primeros 500 caracteres):")
print("=" * 60)
print(page.page_content[0:500])
print("=" * 60)
print("(Contenido truncado para visualización)")

In [ ]:
# Examinamos los metadatos de la página
print("METADATOS DE LA PÁGINA:")
print("=" * 30)
print(page.metadata)
print("=" * 30)
print("Los metadatos incluyen información como número de página y archivo fuente")

## Carga desde la web

WebBaseLoader hace lo mismo con una URL: descarga el HTML, extrae el texto y lo envuelve en un `Document`.

In [ ]:
# Importamos y configuramos el loader para páginas web
from langchain.document_loaders import WebBaseLoader

# Especificamos la URL que queremos procesar
# En este ejemplo usamos el repo de nuestro curso
url = "https://es.wikipedia.org/wiki/Agricultura_en_Argentina"
loader = WebBaseLoader(url)

print(f"WebBaseLoader configurado para: {url}")
print("Listo para extraer contenido web")

In [ ]:
# Cargamos el contenido de la página web
docs = loader.load()

print(f"Contenido web cargado exitosamente: {len(docs)} documento(s)")
print("El HTML ha sido procesado y convertido a texto plano")

In [ ]:
# Mostramos una muestra del contenido extraído de la web
print("CONTENIDO EXTRAÍDO DE LA WEB (primeros 1500 caracteres):")
print("=" * 60)
print(docs[0].page_content[:3000])
print("=" * 60)
print("El contenido web ha sido limpiado y estructurado para procesamiento")

In [ ]:
print("METADATOS DE LA PRIMERA PÁGINA:")
print("=" * 30)
print(docs[0].metadata)
print("=" * 30)
print("Los metadatos incluyen información como número de página y archivo fuente")

### ✎ Para pensar

- Los metadatos incluyen la fuente del documento. ¿Para qué le serviría eso al sistema RAG cuando responde?
- ¿Qué problemas podría tener cargar una página web con mucho JavaScript dinámico?

## Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **Document** | Objeto LangChain con `page_content` + `metadata` |
| **PyPDFLoader** | Carga PDFs página por página |
| **WebBaseLoader** | Extrae texto de una URL |
| **Chunking** | Dividir documentos para que entren en la ventana de contexto del modelo de embeddings |

**Próximo bloque** ChromaDB: vas a convertir estos chunks en vectores y guardarlos en una base de datos que puede buscar por significado.